### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_DOPLVBQUNDZQ


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250522_squid_cal520"
recname = "rec6"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec6"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_yutaka_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_yutaka/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

BAD_CELLS = []
RESCUE_CELLS = []

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(719500,) (10949,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(192, 719500)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.428998,0.318878,0.174222,0.039001,0.198149,0.333765,0.111207,-0.255588,...,-0.482403,0.007098,0.138491,0.087412,0.164227,0.253158,0.149519,0.159963,0.082563,0.105811
1,0,1,0.296567,-0.007864,0.012278,0.185766,0.210307,-0.193807,-0.093662,0.178190,...,0.115669,0.049031,0.017472,-0.440290,-0.231254,0.439819,0.311783,0.081176,0.110492,0.120264
2,0,2,0.309559,0.581043,0.793841,0.914968,0.954606,0.910087,1.079230,0.510880,...,-0.747690,-0.484249,-0.358936,-0.415703,-0.515277,-0.770795,-0.616799,-0.313495,-0.180561,-0.454944
3,0,3,0.351197,0.865445,1.137417,1.165227,0.738696,0.954350,0.906338,0.817524,...,-0.993590,-0.897440,-0.506791,-0.944550,-1.146742,-0.814384,-0.662513,-1.134911,-0.564431,-0.150451
4,0,4,0.437095,0.552177,0.574995,0.636482,0.715729,0.751081,0.683037,0.596698,...,-0.087460,-0.057496,-0.110895,-0.054008,-0.054659,0.011265,-0.104606,-0.121922,0.033568,0.185801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,0,187,0.273754,0.576814,0.268578,0.004038,0.212486,0.774889,0.519236,0.447911,...,-0.472734,-0.891289,-0.688647,-0.669934,-0.688052,-0.696132,-0.699868,-0.948467,-0.741719,-0.827118
188,0,188,0.135177,0.203239,0.787004,0.771219,0.418283,0.112171,-0.219562,-0.331360,...,0.662979,0.455478,0.125729,0.631915,0.551619,0.469112,-0.032424,0.466998,0.035069,0.083911
189,0,189,0.108777,0.226999,0.060144,0.330831,0.602990,0.920282,0.464031,-0.115775,...,0.497857,0.758315,0.168144,-0.074821,0.018053,0.071020,0.099906,-0.357564,-0.221554,-0.314662
190,0,190,0.347754,-0.093924,-0.147579,-0.555467,-0.544272,-0.422625,-0.378776,-0.179353,...,0.458172,0.304318,0.355828,0.564184,0.536815,0.193286,0.467459,0.457679,0.548488,0.406842


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec6"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_DOPLVBQUNDZQ/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec6"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:2
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_DOPLVBQUNDZQ/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec6"', returncode=0)